<h1>LOGISTIC REGRESSION ON CUSTOMER CHURN CLASSIFICATION</h1>

Customer churn refers to the phenomenon where customers discontinue their relationship or subscription with a company or service provider. It represents the rate at which customers stop using a company's products or services within a specific period. Churn is an important metric for businesses as it directly impacts revenue, growth, and customer retention.

In [80]:

import pandas as pd
import zipfile

# the file appears to be a ZIP archive (starts with PK...), open the CSV inside it
with zipfile.ZipFile('customer_churn_dataset-training-master.csv') as z:
	csv_files = [n for n in z.namelist() if n.endswith('.csv')]
	if not csv_files:
		raise ValueError("No CSV file found inside the archive.")
	with z.open(csv_files[0]) as f:
		df = pd.read_csv(f)
df.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1.0
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1.0
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1.0


In [81]:
df.shape

(440833, 12)

In [82]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440833 entries, 0 to 440832
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CustomerID         440832 non-null  float64
 1   Age                440832 non-null  float64
 2   Gender             440832 non-null  object 
 3   Tenure             440832 non-null  float64
 4   Usage Frequency    440832 non-null  float64
 5   Support Calls      440832 non-null  float64
 6   Payment Delay      440832 non-null  float64
 7   Subscription Type  440832 non-null  object 
 8   Contract Length    440832 non-null  object 
 9   Total Spend        440832 non-null  float64
 10  Last Interaction   440832 non-null  float64
 11  Churn              440832 non-null  float64
dtypes: float64(9), object(3)
memory usage: 40.4+ MB


In [83]:
df.isnull().sum()

CustomerID           1
Age                  1
Gender               1
Tenure               1
Usage Frequency      1
Support Calls        1
Payment Delay        1
Subscription Type    1
Contract Length      1
Total Spend          1
Last Interaction     1
Churn                1
dtype: int64

there is one null value in every column <br>
first lets change Gender, Subscription Type, Contract Length into numeric values using lableEncoding 	

In [84]:
from sklearn.preprocessing import LabelEncoder 
le = LabelEncoder()



In [85]:

df['Gender_n'] = le.fit_transform(df['Gender'])
df['Subscription_Type_n'] = le.fit_transform(df['Subscription Type'])
df['Contract Length_n'] = le.fit_transform(df['Contract Length'])

In [86]:
df['Gender_n'].value_counts()

Gender_n
1    250252
0    190580
2         1
Name: count, dtype: int64

In [87]:
df.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn,Gender_n,Subscription_Type_n,Contract Length_n
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0,0,2,0
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0,0,0,1
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1.0,0,0,2
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1.0,1,2,1
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1.0,1,0,1


In [88]:
new_df  = df.drop(['Gender', 'Subscription Type', 'Contract Length'], axis=1)
new_df.head()

,CustomerID,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn,Gender_n,Subscription_Type_n,Contract Length_n
0,2.0,30.0,39.0,14.0,5.0,18.0,932.0,17.0,1.0,0,2,0
1,3.0,65.0,49.0,1.0,10.0,8.0,557.0,6.0,1.0,0,0,1
2,4.0,55.0,14.0,4.0,6.0,18.0,185.0,3.0,1.0,0,0,2
3,5.0,58.0,38.0,21.0,7.0,7.0,396.0,29.0,1.0,1,2,1
4,6.0,23.0,32.0,20.0,5.0,8.0,617.0,20.0,1.0,1,0,1


filling null values 


In [89]:
new_df.mean()

CustomerID             225398.667955
Age                        39.373153
Tenure                     31.256336
Usage Frequency            15.807494
Support Calls               3.604437
Payment Delay              12.965722
Total Spend               631.616223
Last Interaction           14.480868
Churn                       0.567107
Gender_n                    0.567684
Subscription_Type_n         1.013847
Contract Length_n           0.998489
dtype: float64

In [90]:
new_df.fillna(new_df.mean(), inplace=True)


In [91]:

isnull_sum = new_df.isnull().sum()
print(isnull_sum)

CustomerID             0
Age                    0
Tenure                 0
Usage Frequency        0
Support Calls          0
Payment Delay          0
Total Spend            0
Last Interaction       0
Churn                  0
Gender_n               0
Subscription_Type_n    0
Contract Length_n      0
dtype: int64


In [92]:
new_df.head()

,CustomerID,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn,Gender_n,Subscription_Type_n,Contract Length_n
0,2.0,30.0,39.0,14.0,5.0,18.0,932.0,17.0,1.0,0,2,0
1,3.0,65.0,49.0,1.0,10.0,8.0,557.0,6.0,1.0,0,0,1
2,4.0,55.0,14.0,4.0,6.0,18.0,185.0,3.0,1.0,0,0,2
3,5.0,58.0,38.0,21.0,7.0,7.0,396.0,29.0,1.0,1,2,1
4,6.0,23.0,32.0,20.0,5.0,8.0,617.0,20.0,1.0,1,0,1


creating x feature and y target 


In [93]:
x = new_df.drop('Churn', axis=1)
y = new_df['Churn']


In [94]:
y.value_counts()

Churn
1.000000    249999
0.000000    190833
0.567107         1
Name: count, dtype: int64

here it shows that there is one 0.567107 <br>
fixing that value, else it will not consider as classifier it is consider as continueous value  

In [95]:
y = y.round().astype(int)

In [96]:
y.value_counts()

Churn
1    250000
0    190833
Name: count, dtype: int64

applying standard scaling on x 

In [97]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

creating train model amd test model 


In [98]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=42)

In [99]:
len(x_train), len(x_test), len(y_train), len(y_test)

(352666, 88167, 352666, 88167)

model using logistic regression classificaion 

In [100]:
from sklearn.linear_model import LogisticRegression 
model = LogisticRegression()

In [101]:
model.fit(x_train, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [102]:
model.predict(x_test)

array([1, 1, 1, ..., 0, 1, 0], shape=(88167,))

In [104]:
model.score(x_train, y_train)

0.977482944202163

In [103]:
model.score(x_test, y_test)

0.9780870393684712